# C1 — vec2vec-style Demo: Translation Without Paired Examples

**The strongest observable form of representation convergence.**
Experiments A and B used *paired* data (same sentence / same image through
both models) to fit the linear map. This experiment asks the vec2vec /
Platonic question: is the correspondence itself **recoverable from the
shapes of the two spaces alone?**

**Protocol (no pairing used for fitting, ever):**
1. Take N MobileCLIP image vectors and the SigLIP vectors of the *same*
   images — then **shuffle one side and seal the true permutation away**
   (used only for final scoring).
2. **Stage 1 — Gromov-Wasserstein**: match the two *similarity
   structures* (each space's own N x N cosine matrix) to get an initial
   soft correspondence. GW compares shapes to shapes; it never sees a
   cross-space pair.
3. **Stage 2 — ICP refinement**: fit a ridge map from the soft
   correspondence -> re-match by nearest neighbor in the target space ->
   refit; iterate. (The classical unsupervised-alignment loop from
   MUSE-style word translation, applied to image encoders.)
4. **Score**: matching accuracy vs the 1/N chance floor, and retrieval
   through the unsupervised map vs the supervised ridge map from B2.

**Honesty note:** real vec2vec (Jha et al. 2025) uses adversarial +
cycle-consistency training between *different corpora* (not even the
same items). This demo is the transparent classical version of the same
claim on shared items with hidden correspondence — the appropriate
scope for a course project, and the logic is identical: if shapes were
not nearly identical, none of this could work.

Needs: `pairs.npz`. Runtime: ~3-6 min CPU (GW is the slow part).


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
# 2. Load, subsample, and DESTROY the pairing
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ['DATA_DIR'])
pairs = np.load(DATA_DIR / 'pairs.npz')

N = 1500                                   # GW is O(N^2); 1500 is comfy
rng = np.random.default_rng(0)
sub = rng.choice(len(pairs['mob_img']), N, replace=False)

A = pairs['mob_img'][sub].astype(np.float64)      # MobileCLIP, 512-d
B_true = pairs['sig_img'][sub].astype(np.float64) # SigLIP,    768-d

perm = rng.permutation(N)                  # the shuffle
B = B_true[perm]                           # B[i] is NOT A[i]'s partner
# ground truth, SEALED: A[i]'s true partner sits at B[inv_perm[i]]
inv_perm = np.empty(N, dtype=int); inv_perm[perm] = np.arange(N)

print(f'{N} vectors per side; pairing destroyed.')
print(f'chance matching accuracy = 1/{N} = {1/N:.4%}')

In [ ]:
# 3. Stage 1 - Gromov-Wasserstein: match shape to shape
# Each space described ONLY by its internal cosine-distance matrix.
import ot, time

CA = 1.0 - A @ A.T                         # within-space distances
CB = 1.0 - B @ B.T
CA /= CA.mean(); CB /= CB.mean()           # scale-normalize the shapes

p = np.full(N, 1/N); q = np.full(N, 1/N)
t0 = time.time()
G = ot.gromov.entropic_gromov_wasserstein(
    CA, CB, p, q, loss_fun='square_loss', epsilon=5e-4,
    max_iter=200, tol=1e-9, verbose=False)
print(f'GW done in {time.time()-t0:.0f}s')

gw_match = G.argmax(axis=1)                # hard assignment per A-row
gw_acc = (gw_match == inv_perm).mean()
print(f'GW matching accuracy: {gw_acc:.1%}  (chance {1/N:.3%})')

In [ ]:
# 4. Stage 2 - ICP refinement: fit map -> re-match -> refit
def ridge(X, Y, a=1e-2):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + a * np.eye(d), X.T @ Y)

def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

match = gw_match.copy()
for it in range(15):
    W_u = ridge(A, B[match])               # fit on CURRENT belief
    proj = l2n(A @ W_u)
    sims = proj @ B.T                      # nearest neighbor re-match
    new_match = sims.argmax(axis=1)
    acc = (new_match == inv_perm).mean()
    changed = (new_match != match).mean()
    print(f'iter {it:2d}: matching accuracy {acc:6.1%}   '
          f'({changed:.1%} assignments changed)')
    if (new_match == match).all():
        break
    match = new_match

final_acc = (match == inv_perm).mean()
# top-10 tolerance: true partner within the 10 nearest
top10 = (-sims).argsort(axis=1)[:, :10]
top10_acc = (top10 == inv_perm[:, None]).any(1).mean()
print(f'\nFinal: exact matching {final_acc:.1%}, '
      f'true partner in top-10 {top10_acc:.1%}  (chance {1/N:.3%})')

In [ ]:
# 5. The payoff: retrieval through the UNSUPERVISED map vs the
#    supervised map from B2, on the standard held-out protocol
ad = np.load(DATA_DIR / 'adapter.npz')
te = ad['eval_idx']
W_sup = ad['W_ridge'].astype(np.float64)

txt = pairs['sig_txt'][te]
sig = pairs['sig_img'][te]
mob = pairs['mob_img'][te]

def recall(sim, ks=(1, 5, 10)):
    r = (-sim).argsort(axis=1)
    n = sim.shape[0]
    return {k: float((r[:, :k] == np.arange(n)[:, None]).any(1).mean())
            for k in ks}

def report(name, r, ceil=None):
    line = f'{name:<44} ' + '  '.join(f'R@{k}={r[k]:.3f}' for k in (1,5,10))
    if ceil:
        pct = min(100*r[k]/max(ceil[k],1e-9) for k in (1,5,10))
        line += f'  (worst-K {pct:.0f}% of ceiling)'
    print(line)

ceil = recall(txt @ sig.T)
r_sup = recall(txt @ l2n(mob @ W_sup).T)
r_uns = recall(txt @ l2n(mob @ W_u).T)
report('SigLIP native (ceiling)', ceil)
report('supervised W (B2 - 3,000 true pairs)', r_sup, ceil)
report('UNSUPERVISED W (0 pairs, geometry only)', r_uns, ceil)

## How to read the result

- **Matching accuracy**: exact-match far above 1/1500 = 0.067% chance is
  already the finding — the correspondence between two vendors' spaces
  is encoded in their shapes. The ICP curve typically shows the
  characteristic *avalanche*: mediocre GW seed -> each refit sharpens
  the map -> sharper map fixes more matches -> convergence.
- **The retrieval table** is the headline: if the unsupervised map lands
  anywhere near the supervised one, you have reproduced, at course
  scale, the vec2vec claim - *translation between embedding spaces
  without a single paired example* - and with it the strongest
  observable form of the Platonic Representation Hypothesis.
- **If GW seeds poorly** (accuracy ~0 after stage 1): raise `epsilon`
  to 1e-3, or drop N to 1000; the ICP stage is robust to a weak seed as
  long as some signal exists.

## The security corollary (state it in the report)
This same procedure is why embeddings must be treated as content, not
anonymization: anyone holding your "opaque" vectors plus *any* strong
encoder of the same modality can run exactly this recovery. Your
privacy-preserving indexing story stays honest because it never claimed
embeddings are unreadable - only that raw photos never leave the device.
